Merkle trees are used in Bitcoin because:

1. It allows you check if a transaction is included in the blockchain, yourself. 
2. And, you can do this with less data.. you don't have to download all the blocks & transactions (800GB on Nov 2026) and check each one by one for inclusion. Instead, you can download just the blocks with headers (~78MB), and prove that a transaction exists in a block with the help of merkle trees.

To make Merkle trees work, we need to have:
1. An ordered list.
2. A hash function.

In Bitcoin, the ordered list is the list of transactions included in the block, and the hash function is SHA256. The basic idea is to hash a pair of transactions, to create the merkle parent.

In [5]:
import math
import sys
from importlib import reload
from io import BytesIO
from pathlib import Path

# Use the functions from Programming Bitcoin, Chapter 11.
chapter_dir = str(Path("../programmingbitcoin/book/code-ch11").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

import block
import ecc
import helper
import merkleblock
import network
import script
import tx

from block import Block
from helper import (
    bytes_to_bit_field,
    hash256,
    little_endian_to_int,
    merkle_parent,
    merkle_parent_level,
    merkle_root,
    read_varint,
    run,
)
from merkleblock import MerkleBlock, MerkleTree
from network import GetHeadersMessage, HeadersMessage, SimpleNode
from tx import Tx, TxFetcher

tr_1 = bytes.fromhex('c117ea8ec828342f4dfb0ad6bd140e03a50720ece40169ee38bdc15d9eb64cf5')
tr_2 = bytes.fromhex('c131474164b412e3406696da1ee20ab0fc9bf41c8f05fa8ceea7a08d672d7cc5')
parent = hash256(tr_1 + tr_2)
print(parent.hex())

8b30c5ba100f6f2e5ad1e2a742e5020491240f8eb514fe97c713c31718ad7ecd


Since the parent is a combination of two children, the parent row will always have half the number of hashes of the children row. We can then continue hashing the merkle parent rows, until we are left with just one single hash in a row, called the merkle root. The merkle root is what is included in the block header, and a light client can prove or verify that a transaction is included by requesting only the sibling hashes of each row from the full node.

This is an example of a merkle tree with 10 items:

In [19]:
total = 10
max_depth = math.ceil(math.log(total, 2))
merkle_tree = []
for depth in range(max_depth + 1):
    re = 2**(max_depth - depth)
    num_items = math.ceil(total / 2**(max_depth - depth))
    level_hashes = [None] * num_items
    merkle_tree.append(level_hashes)
for level in merkle_tree:
    print(level)

[None]
[None, None]
[None, None, None]
[None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None]


In [21]:
hex_hashes = [
    "9745f7173ef14ee4155722d1cbf13304339fd00d900b759c6f9d58579b5765fb",
    "5573c8ede34936c29cdfdfe743f7f5fdfbd4f54ba0705259e62f39917065cb9b",
    "82a02ecbb6623b4274dfcab82b336dc017a27136e08521091e443e62582e8f05",
    "507ccae5ed9b340363a0e6d765af148be9cb1c8766ccc922f83e4ae681658308",
    "a7a4aec28e7162e1e9ef33dfa30f0bc0526e6cf4b11a576f6c5de58593898330",
    "bb6267664bd833fd9fc82582853ab144fece26b7a8a5bf328f8a059445b59add",
    "ea6d7ac1ee77fbacee58fc717b990c4fcccf1b19af43103c090f601677fd8836",
    "457743861de496c429912558a106b810b0507975a49773228aa788df40730d41",
    "7688029288efc9e9a0011c960a6ed9e5466581abf3e3a6c26ee317461add619a",
    "b1ae7f15836cb2286cdd4e2c37bf9bb7da0a2846d06867a429f654b2e7f383c9",
]
tree = MerkleTree(len(hex_hashes))
last_level = len(tree.nodes) - 1

tree.nodes[last_level] = [bytes.fromhex(h) for h in hex_hashes]

for depth in range(last_level, 0, -1):
    tree.nodes[depth - 1] = merkle_parent_level(
        tree.nodes[depth].copy()
    )

print(tree)

*14c4eea1.*
6382df3f..., a8cef966...
3ba6c080..., 8e894862..., 8c65546c...
272945ec..., 9a38d037..., 4a64abd9..., ec7c95e1..., 3b67006c...
9745f717..., 5573c8ed..., 82a02ecb..., 507ccae5..., a7a4aec2..., bb626766..., ea6d7ac1..., 45774386..., 76880292..., b1ae7f15...


Next, let's look at a random block, and then check if a transaction is included in it. I've already connected to the Bitcoin server, and got these details via the cli:



Start Bitcoin Core:

```bash
/usr/local/bin/bitcoind -daemon -peerbloomfilters=1
```

```text
Bitcoin Core starting
```

Get the latest block hash:

```bash
/usr/local/bin/bitcoin-cli getbestblockhash
```

```text
0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50
```

Read the block details and select its first transaction:

```bash
/usr/local/bin/bitcoin-cli getblock "0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50" 1 |
/usr/bin/python3 -c '
import json, sys

block = json.load(sys.stdin)

print("Block height:", block["height"])
print("Block hash:", block["hash"])
print("Merkle root:", block["merkleroot"])
print("Transaction count:", block["nTx"])
print("Transaction ID:", block["tx"][0])
'
```

```text
Block height: 888687
Block hash: 0000000000000000000048a42e44ef87555b7d6729fd8b5b53f17a0469379c50
Merkle root: 0a61e391dd9ce8f3c10e73c1b9624ccbac5b04f4450fd45e26d1b18324c6dc6f
Transaction count: 3129
Transaction ID: f13de3703e09f77b03a1a184beb9ffcfd108ba1a5718daa664d3292b5b495ad5
```

We can check if `f13de3703e09f77b03a1a184beb9ffcfd108ba1a5718daa664d3292b5b495ad5` is included in block 888,687.

Let's start with a handshake:

In [24]:
import sys

chapter_dir = "/Users/arjunrajkumar/Developer/programmingbitcoin/code-ch11"
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

from network import SimpleNode

node = SimpleNode(
    "127.0.0.1",
    port=8333,
    testnet=False,
    logging=True,
)

node.socket.settimeout(15)

# Use buffered reads to receive complete messages.
node.stream.close()
node.stream = node.socket.makefile("rb")

node.handshake()
print("Handshake complete")

sending: version: 7f11010000000000000000008420096b00000000000000000000000000000000000000000000ffff00000000208d000000000000000000000000000000000000ffff00000000208d35d95f0a20c6f5c7182f70726f6772616d6d696e67626974636f696e3a302e312f0000000000
receiving: version: 801101000c040000000000008420096b0000000000000000000000000000000000000000000000000000000000000c04000000000000000000000000000000000000000000000000a94f1ca26d23ca02102f5361746f7368693a32342e302e312f87940d0001
sending: verack: 
receiving: verack: 
Handshake complete


Next, let's send a Bloom filter for telling the node which transaction we want.

In [25]:
from helper import encode_varint


class RawMessage:
    def __init__(self, command, payload):
        self.command = command
        self.payload = payload

    def serialize(self):
        return self.payload


txid = "f13de3703e09f77b03a1a184beb9ffcfd108ba1a5718daa664d3292b5b495ad5"

# Reverse the bytes for Bitcoin's network format.
tx_hash = bytes.fromhex(txid)[::-1]

# Create an empty filter.
filter_bytes = b"\x00" * 100
filter_payload = (
    encode_varint(len(filter_bytes))
    + filter_bytes
    + (5).to_bytes(4, "little")   # Number of filter hash functions.
    + (99).to_bytes(4, "little")  # Value used to vary those functions.
    + b"\x00"                   # Disable automatic filter updates.
)

node.send(RawMessage(b"filterload", filter_payload))

# Add the transaction hash to the filter.
node.send(
    RawMessage(b"filteradd", encode_varint(len(tx_hash)) + tx_hash)
)

print("Filter messages sent")

sending: filterload: 6400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000050000006300000000
sending: filteradd: 20d55a495b2b29d364a6da18571aba08d1cfffb9be84a1a1037bf7093e70e33df1
Filter messages sent


Next, let's request the merkleblock.

### Get a Merkle proof in Terminal

These steps use Bitcoin Core's local command interface (RPC). The proof is copied into the next code cell for use in the notebook. This method returns the proof data. It does not capture the full peer-to-peer message exchange.

#### 1. Choose a recent block and read its header

Run this in Terminal. It selects a random block from the latest 20 blocks held by the node.

```bash
proof_tip=$(/usr/local/bin/bitcoin-cli getblockcount)

proof_height=$((proof_tip - RANDOM % 20))

proof_hash=$(/usr/local/bin/bitcoin-cli getblockhash "$proof_height")

/usr/local/bin/bitcoin-cli getblockheader "$proof_hash"
```

For this run, the selected block had these values:

```text
Block height: 890946
Block hash: 000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc
Merkle root: 5d9e3af145efd681e5b79720d67600bd6172a238aa03ce5e31dee48909beb938
Transaction count: 139
```

The header contains the Merkle root. Reading the header does not download the full block.

#### 2. Save the full block data and select a transaction

The `2` requests details for every transaction. This command reads the block from the local node and saves the returned JSON data.

```bash
/usr/local/bin/bitcoin-cli getblock "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc" 2 |
/usr/bin/python3 -c '
import json
import sys
from pathlib import Path

block = json.load(sys.stdin)

path = Path.home() / "Downloads" / "bitcoin-block-890946.json"
path.write_text(json.dumps(block, indent=2) + "\n")

print("Saved full block data to:", path)
print("Block height:", block["height"])
print("Transaction count:", block["nTx"])
print("Transaction ID:", block["tx"][0]["txid"])
'
```

Output:

```text
Saved full block data to: /Users/arjunrajkumar/Downloads/bitcoin-block-890946.json
Block height: 890946
Transaction count: 139
Transaction ID: 89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3
```

#### 3. Request the proof for that transaction

Run this in Terminal. `gettxoutproof` returns the proof as hexadecimal text. The block hash tells the node which block to use. `tee` prints the proof and saves a copy.

```bash
/usr/local/bin/bitcoin-cli gettxoutproof \
  '["89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3"]' \
  "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc" \
  | tee "$HOME/Downloads/bitcoin-proof-890946.hex"
```

The returned proof contains a block header, the total transaction count, proof hashes, and flag bytes. The complete proof text is in the next cell. That cell parses the proof into fields; parsing alone does not verify transaction inclusion.

Command references: [getblockheader](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/getblockheader/), [getblock](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/getblock/), and [gettxoutproof](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/gettxoutproof/).


In [26]:
import sys
from io import BytesIO
from pathlib import Path

# Use the Chapter 11 parser from the local book checkout.
chapter_dir = str(Path("../programmingbitcoin/book/code-ch11").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

from merkleblock import MerkleBlock

# Keep the selected block and transaction for the later proof check.
block_hash = "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc"
txid = "89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3"
expected_merkle_root = "5d9e3af145efd681e5b79720d67600bd6172a238aa03ce5e31dee48909beb938"

# Proof returned by bitcoin-cli gettxoutproof.
proof_hex = (
    "00a00e20c31a2ffc2e001d2979da36bf4ff2b2af262d0cded6cc0100000000000000000038b9be0989e4de315ece03aa"
    "38a27261bd0076d62097b7e581d6ef45f13a9e5d0f53f0676c7902176b353f058b00000009b350ce70d0bc25e3d20824"
    "d5378d350ccfee48199655162d4bedf09db60f668921d4fa09f669e70286c7efe6d8aac76b582c9bf2dbf7b653fde443"
    "a8dd889602a97ea4ca6ce9b6f563dd4373f5bd12213ebc2dd370bea4709d00da19471f06834182c9475ae0045061d04a"
    "66d8812c2f1f519ceb587812384eb92a60f6bd6712b456e6af0c71a16824fbcf784d5822630798bb102ad015369b28e7"
    "992a5240a8b933fc9756ab501e49d0c966a701d2358b43508f7f2b47bd79a37c4e94e8aac7d4bf5e33741a4bd36ffb2d"
    "fba368e0f498f4192a56af96f03d657a79e27f463c2e000f3fb5c9d955570e27209d2bc43a92ba4ab57de101d7ce771f"
    "cfa5739a251329de31658e6821e48e4051efe787a5cac089f90613188a2d354c4f4569f2e703ff0100"
)

# Parse the saved text. This does not connect to Bitcoin Core.
proof_bytes = bytes.fromhex(proof_hex)
proof_stream = BytesIO(proof_bytes)
merkle_block = MerkleBlock.parse(proof_stream)

print("Proof size (bytes):", len(proof_bytes))
print("Transaction count:", merkle_block.total)
print("Merkle root:", merkle_block.merkle_root.hex())
print("Proof hash count:", len(merkle_block.hashes))
print("Flags:", merkle_block.flags.hex())
print("Unread bytes:", len(proof_stream.read()))

for index, proof_hash in enumerate(merkle_block.hashes):
    print(f"Hash {index}:", proof_hash.hex())


NotImplementedError: 